In [1]:
import torch
import torch.nn as nn

In [2]:
#-------------------------- PATCH EMBEDDING--------------------------#

class PatchEmbed(nn.Module):
    def __init__(self, img_size=224, patch_size=4, in_chans=3, embed_dim=96, norm_layer=None):
        super().__init__()
        
        self.img_size = (img_size, img_size)
        self.patch_size = (patch_size, patch_size)
        self.num_patches = (img_size // patch_size) ** 2  # 56*56 = 3136
        self.embed_dim = embed_dim
        
        # one conv does patch splitting + embedding in one shot
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        
        # optional normalization
        self.norm = norm_layer(embed_dim) if norm_layer else None

    def forward(self, x):
        B, C, H, W = x.shape
        
        x = self.proj(x)        # (B, 96, 56, 56)
        x = x.flatten(2)        # (B, 96, 3136)
        x = x.transpose(1, 2)  # (B, 3136, 96)
        
        if self.norm:
            x = self.norm(x)
            
        return x

In [3]:
x = torch.randn(1, 3, 224, 224)
patch_embed = PatchEmbed()
out = patch_embed(x)
print(out.shape)  # should print torch.Size([1, 3136, 96])

torch.Size([1, 3136, 96])


In [4]:
def window_partition(x, window_size):
    B, H, W, C = x.shape
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)
    return windows

In [5]:
def window_reverse(windows, window_size, H, W):
    B = int(windows.shape[0] / (H * W / window_size / window_size))
    x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)
    return x

In [6]:
x = torch.randn(1, 56, 56, 96)
windows = window_partition(x, window_size=7)
print(windows.shape)  # should be torch.Size([64, 7, 7, 96])

torch.Size([64, 7, 7, 96])


In [7]:
class WindowAttention(nn.Module):
    def __init__(self, dim, window_size, num_heads, qkv_bias=True, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.dim = dim
        self.window_size = window_size  # 7
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5

        # relative position bias table
        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window_size - 1) * (2 * window_size - 1), num_heads))
        nn.init.trunc_normal_(self.relative_position_bias_table, std=.02)

        # relative position index
        coords_h = torch.arange(self.window_size)
        coords_w = torch.arange(self.window_size)
        coords = torch.stack(torch.meshgrid([coords_h, coords_w], indexing='ij'))  # 2, 7, 7
        coords_flatten = torch.flatten(coords, 1)  # 2, 49

        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]  # 2, 49, 49
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()  # 49, 49, 2
        relative_coords[:, :, 0] += self.window_size - 1
        relative_coords[:, :, 1] += self.window_size - 1
        relative_coords[:, :, 0] *= 2 * self.window_size - 1
        relative_position_index = relative_coords.sum(-1)  # 49, 49
        self.register_buffer("relative_position_index", relative_position_index)

        self.qkv = nn.Linear(dim, dim * 3, qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x, mask=None):
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)

        q = q * self.scale
        attn = (q @ k.transpose(-2, -1))

        # add relative position bias
        relative_position_bias = self.relative_position_bias_table[
            self.relative_position_index.view(-1)].view(
            self.window_size * self.window_size,
            self.window_size * self.window_size, -1)
        relative_position_bias = relative_position_bias.permute(2, 0, 1).contiguous()
        attn = attn + relative_position_bias.unsqueeze(0)

        if mask is not None:
            nW = mask.shape[0]
            attn = attn.view(B_ // nW, nW, self.num_heads, N, N) + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)

        attn = self.softmax(attn)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

In [8]:
attn = WindowAttention(dim=96, window_size=7, num_heads=3)
x = torch.randn(64, 49, 96)  # 64 windows, 49 tokens per window (7x7), 96 dim
out = attn(x)
print(out.shape)  # should be torch.Size([64, 49, 96])

torch.Size([64, 49, 96])


In [9]:
from timm.layers import DropPath

In [10]:
class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, drop=0.):
        super().__init__()
        hidden_features = hidden_features or in_features
        out_features = out_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x


class SwinTransformerBlock(nn.Module):
    def __init__(self, dim, input_resolution, num_heads, window_size=7,
                 shift_size=0, mlp_ratio=4., qkv_bias=True,
                 drop=0., attn_drop=0., drop_path=0.):
        super().__init__()
        self.dim = dim
        self.input_resolution = input_resolution
        self.num_heads = num_heads
        self.window_size = window_size
        self.shift_size = shift_size

        # if window is larger than input, no shifting needed
        if min(self.input_resolution) <= self.window_size:
            self.shift_size = 0
            self.window_size = min(self.input_resolution)

        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention(
            dim, window_size=self.window_size,
            num_heads=num_heads,
            qkv_bias=qkv_bias,
            attn_drop=attn_drop,
            proj_drop=drop)

        self.drop_path = nn.Identity() if drop_path == 0. else DropPath(drop_path)
        self.norm2 = nn.LayerNorm(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, drop=drop)

        # attention mask for shifted windows
        if self.shift_size > 0:
            H, W = self.input_resolution
            img_mask = torch.zeros((1, H, W, 1))
            h_slices = (slice(0, -self.window_size),
                        slice(-self.window_size, -self.shift_size),
                        slice(-self.shift_size, None))
            w_slices = (slice(0, -self.window_size),
                        slice(-self.window_size, -self.shift_size),
                        slice(-self.shift_size, None))
            cnt = 0
            for h in h_slices:
                for w in w_slices:
                    img_mask[:, h, w, :] = cnt
                    cnt += 1
            mask_windows = window_partition(img_mask, self.window_size)
            mask_windows = mask_windows.view(-1, self.window_size * self.window_size)
            attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
            attn_mask = attn_mask.masked_fill(attn_mask != 0, -100.0).masked_fill(attn_mask == 0, 0.0)
        else:
            attn_mask = None

        self.register_buffer("attn_mask", attn_mask)

    def forward(self, x, H, W):
        B, L, C = x.shape

        shortcut = x
        x = self.norm1(x)
        x = x.view(B, H, W, C)

        # cyclic shift
        if self.shift_size > 0:
            shifted_x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
        else:
            shifted_x = x

        # partition windows
        x_windows = window_partition(shifted_x, self.window_size)
        x_windows = x_windows.view(-1, self.window_size * self.window_size, C)

        # attention
        attn_windows = self.attn(x_windows, mask=self.attn_mask)

        # merge windows
        attn_windows = attn_windows.view(-1, self.window_size, self.window_size, C)
        shifted_x = window_reverse(attn_windows, self.window_size, H, W)

        # reverse cyclic shift
        if self.shift_size > 0:
            x = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))
        else:
            x = shifted_x

        x = x.view(B, H * W, C)
        x = shortcut + self.drop_path(x)

        # MLP
        x = x + self.drop_path(self.mlp(self.norm2(x)))

        return x

In [11]:
block = SwinTransformerBlock(
    dim=96,
    input_resolution=(56, 56),
    num_heads=3,
    window_size=7,
    shift_size=0
)
x = torch.randn(1, 56*56, 96)
out = block(x, 56, 56)
print(out.shape)  # torch.Size([1, 3136, 96])

torch.Size([1, 3136, 96])


In [12]:
block_shifted = SwinTransformerBlock(
    dim=96,
    input_resolution=(56, 56),
    num_heads=3,
    window_size=7,
    shift_size=3  # shifted window
)
x = torch.randn(1, 56*56, 96)
out = block_shifted(x, 56, 56)
print(out.shape)  # should still be torch.Size([1, 3136, 96])

torch.Size([1, 3136, 96])


In [13]:
class PatchMerging(nn.Module):
    def __init__(self, input_resolution, dim, norm_layer=nn.LayerNorm):
        super().__init__()
        self.input_resolution = input_resolution
        self.dim = dim
        self.reduction = nn.Linear(4 * dim, 2 * dim, bias=False)
        self.norm = norm_layer(4 * dim)

    def forward(self, x, H, W):
        B, L, C = x.shape
        x = x.view(B, H, W, C)

        # grab 2x2 neighboring patches
        x0 = x[:, 0::2, 0::2, :]  # top left
        x1 = x[:, 1::2, 0::2, :]  # bottom left
        x2 = x[:, 0::2, 1::2, :]  # top right
        x3 = x[:, 1::2, 1::2, :]  # bottom right

        x = torch.cat([x0, x1, x2, x3], -1)  # B, H/2, W/2, 4C
        x = x.view(B, -1, 4 * C)

        x = self.norm(x)
        x = self.reduction(x)  # B, H/2*W/2, 2C

        return x

In [14]:
patch_merging = PatchMerging(input_resolution=(56, 56), dim=96)
x = torch.randn(1, 56*56, 96)
out = patch_merging(x, 56, 56)
print(out.shape)  # should be torch.Size([1, 784, 192])

torch.Size([1, 784, 192])


In [15]:
class BasicLayer(nn.Module):
    def __init__(self, dim, input_resolution, depth, num_heads, window_size,
                 mlp_ratio=4., qkv_bias=True, drop=0., attn_drop=0.,
                 drop_path=0., downsample=None):
        super().__init__()
        self.dim = dim
        self.input_resolution = input_resolution
        self.depth = depth

        # stack multiple SwinTransformerBlocks
        self.blocks = nn.ModuleList([
            SwinTransformerBlock(
                dim=dim,
                input_resolution=input_resolution,
                num_heads=num_heads,
                window_size=window_size,
                shift_size=0 if (i % 2 == 0) else window_size // 2,  # alternate W-MSA and SW-MSA
                mlp_ratio=mlp_ratio,
                qkv_bias=qkv_bias,
                drop=drop,
                attn_drop=attn_drop,
                drop_path=drop_path[i] if isinstance(drop_path, list) else drop_path
            )
            for i in range(depth)
        ])

        # downsampling at end of stage
        self.downsample = downsample

    def forward(self, x, H, W):
        for blk in self.blocks:
            x = blk(x, H, W)

        if self.downsample is not None:
            x = self.downsample(x, H, W)
            H, W = H // 2, W // 2

        return x, H, W

In [16]:
layer = BasicLayer(
    dim=96,
    input_resolution=(56, 56),
    depth=2,
    num_heads=3,
    window_size=7,
    downsample=PatchMerging(input_resolution=(56, 56), dim=96)  # instantiate it here
)
x = torch.randn(1, 56*56, 96)
out, H, W = layer(x, 56, 56)
print(out.shape)  # torch.Size([1, 784, 192])
print(H, W)       # 28 28

torch.Size([1, 784, 192])
28 28


In [17]:
class SwinTransformer(nn.Module):
    def __init__(self, img_size=224, patch_size=4, in_chans=3, num_classes=7,
                 embed_dim=96, depths=[2, 2, 18, 2], num_heads=[3, 6, 12, 24],
                 window_size=7, mlp_ratio=4., qkv_bias=True,
                 drop_rate=0., attn_drop_rate=0., drop_path_rate=0.1):
        super().__init__()
        self.num_classes = num_classes
        self.num_layers = len(depths)
        self.embed_dim = embed_dim
        self.mlp_ratio = mlp_ratio

        # patch embedding
        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size,
            in_chans=in_chans, embed_dim=embed_dim)

        self.pos_drop = nn.Dropout(p=drop_rate)

        # stochastic depth decay rule
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]

        # build 4 stages
        self.layers = nn.ModuleList()
        for i_layer in range(self.num_layers):
            dim = int(embed_dim * 2 ** i_layer)
            input_resolution = (
                img_size // patch_size // (2 ** i_layer),
                img_size // patch_size // (2 ** i_layer)
            )
            depth = depths[i_layer]
            downsample = PatchMerging(
                input_resolution=input_resolution,
                dim=dim
            ) if i_layer < self.num_layers - 1 else None

            layer = BasicLayer(
                dim=dim,
                input_resolution=input_resolution,
                depth=depth,
                num_heads=num_heads[i_layer],
                window_size=window_size,
                mlp_ratio=mlp_ratio,
                qkv_bias=qkv_bias,
                drop=drop_rate,
                attn_drop=attn_drop_rate,
                drop_path=dpr[sum(depths[:i_layer]):sum(depths[:i_layer+1])],
                downsample=downsample
            )
            self.layers.append(layer)

        # final norm + classifier head
        self.norm = nn.LayerNorm(int(embed_dim * 2 ** (self.num_layers - 1)))
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Linear(int(embed_dim * 2 ** (self.num_layers - 1)), num_classes)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        # patch embedding
        x = self.patch_embed(x)
        x = self.pos_drop(x)

        H, W = 56, 56  # initial resolution after patch embed

        # pass through 4 stages
        for layer in self.layers:
            x, H, W = layer(x, H, W)

        x = self.norm(x)
        x = self.avgpool(x.transpose(1, 2))
        x = torch.flatten(x, 1)
        x = self.head(x)

        return x

In [18]:
model = SwinTransformer(
    img_size=224,
    num_classes=7,
    embed_dim=96,
    depths=[2, 2, 18, 2],    # Swin-S config
    num_heads=[3, 6, 12, 24]  # Swin-S config
)

x = torch.randn(1, 3, 224, 224)
out = model(x)
print(out.shape)  # torch.Size([1, 7])

# verify parameter count
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params/1e6:.1f}M")  # should be ~50M

torch.Size([1, 7])
Total parameters: 48.8M
